# Optional advanced practicum: fine-tune LingBot-VLA-v2 with RoboTwin data

This notebook keeps the 43-cell technical coverage of the path-validated competition reference while organizing it as a course. The notebook kernel must have the current `robo_genesis` package installed; GPU-heavy commands are then launched through `/opt/robotwin-env/bin/python`. The required route is: preflight → base serving/evaluation → 4-GPU, 100-step action-expert LoRA → DCP-to-Hugging-Face merge → merged serving/evaluation. Full-parameter SFT, official post-trained checkpoint evaluation, 1,000-episode benchmarks, and the 8-GPU profile remain visible opt-in extensions.

The current pinned companion reports aggregate `k/n` but cannot recover per-resolved-seed policy outcomes. That does **not** block training or serving. This notebook therefore reports independent aggregate observations and Wilson intervals; it does not claim resolved-seed-matched improvement. No GPU experiment in this notebook is presented as already run by RoboGenesis.


In [ ]:
from pathlib import Path
import ast, json, os, signal, subprocess, time, urllib.request

from robo_genesis.l14_workflow import (
    COMPANION_COMMIT, ROBOTWIN_COMMIT,
    archive_changed_files,
    audit_hf_checkpoint, audit_required_paths,
    build_aggregate_evidence_card, build_single_task_eval_command,
    command_text, l14_output_paths, parse_final_success_rate,
    require_fresh_paths, run_command_logged, snapshot_files,
    validate_l14_config,
    write_json_artifact,
)

CONFIG = validate_l14_config({
    'run_name': 'l14-adjust-bottle-4gpu-001',  # Change for every independent run.
    'robotwin_root': '/RoboTwin',
    'runtime_root': '/workspace/runtime',
    'python': '/opt/robotwin-env/bin/python',
    'task_name': 'adjust_bottle',
    'task_config': 'demo_clean',
    'core_gpu_count': 4,
    'competition_gpu_count': 8,
    'lora_steps': 100,
    'eval_episodes': 10,
    'training_seed': 42,
})
ROBOTWIN = Path(CONFIG['robotwin_root'])
RUNTIME = Path(CONFIG['runtime_root'])
PYTHON = CONFIG['python']
PATHS = l14_output_paths(CONFIG)
TRAINING_ROOT = ROBOTWIN / 'experiments/lingbot_vla_v2_6b_robotwin'
SOURCE_ROOT = TRAINING_ROOT / 'source/lingbot-vla-v2'
MODEL = TRAINING_ROOT / 'models/robbyant_lingbot-vla-v2-6b'
LINGBOT_VLA_COMMIT = '951475ae1b1d87553e7dc47c97b53a3d695c0d13'
PORT = 13400
EVAL_RESULT_ROOT = RUNTIME / 'eval_result'

os.environ.setdefault('AITER_TRITON_ONLY', '1')
os.environ.setdefault('FLASH_ATTENTION_TRITON_AMD_ENABLE', 'TRUE')
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
os.environ['TMPDIR'] = str(RUNTIME / 'tmp')
os.environ['PYTHONPATH'] = '/opt/aiter' + (
    ':' + os.environ['PYTHONPATH'] if os.environ.get('PYTHONPATH') else ''
)
print(json.dumps(CONFIG, indent=2, sort_keys=True))
print('Expected companion commit:', COMPANION_COMMIT)
print('Expected RoboTwin commit:', ROBOTWIN_COMMIT)


## 1. Environment, data, and identity preflight

Run the preflight once before allocating model memory. It verifies the pinned repositories, four selected W7900 devices, the 50-dataset training manifest, three camera streams, LoRA configuration, normalization inputs, and fresh output identity. A failure here is a stop condition, not a reason to silently switch hardware or data.


In [ ]:
required = {
    'objaverse index': ROBOTWIN / 'assets/objects/objaverse/list.json',
    'RoboTwin demos': ROBOTWIN / 'data/demo_clean',
    'LeRobot datasets': ROBOTWIN / 'data/lerobot',
    'training dataset list': ROBOTWIN / 'data/robotwin_demo_clean_joint_v30.txt',
    'base weight index': MODEL / 'model.safetensors.index.json',
    'LoRA training YAML': TRAINING_ROOT / 'training/lingbotvla_cli.yaml',
    'normalization stats': SOURCE_ROOT / 'assets/norm_stats/robotwin.json',
}
input_audit = audit_required_paths(required)
require_fresh_paths([PATHS['run_root']])

companion_checkout = Path('/workspace/Robotwin-radeon-cloud')
companion_checkout_verified = (companion_checkout / '.git').exists()
companion_commit = subprocess.check_output(
    ['git', '-C', str(companion_checkout), 'rev-parse', 'HEAD'], text=True
).strip() if companion_checkout_verified else COMPANION_COMMIT
robotwin_commit = subprocess.check_output(
    ['git', '-C', str(ROBOTWIN), 'rev-parse', 'HEAD'], text=True
).strip()
lingbot_commit = subprocess.check_output(
    ['git', '-C', str(SOURCE_ROOT), 'rev-parse', 'HEAD'], text=True
).strip()
assert companion_commit == COMPANION_COMMIT, (companion_commit, COMPANION_COMMIT)
assert robotwin_commit == ROBOTWIN_COMMIT, (robotwin_commit, ROBOTWIN_COMMIT)
assert lingbot_commit == LINGBOT_VLA_COMMIT, (lingbot_commit, LINGBOT_VLA_COMMIT)

gpu_probe = subprocess.check_output([
    PYTHON, '-c',
    "import json, torch; print(json.dumps({'torch': torch.__version__, 'hip': torch.version.hip, 'available': torch.cuda.is_available(), 'count': torch.cuda.device_count(), 'names': [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]}))",
], text=True)
gpu = json.loads(gpu_probe.strip().splitlines()[-1])
assert gpu['hip'], 'PyTorch must report a HIP runtime'
assert gpu['available'] and gpu['count'] >= CONFIG['core_gpu_count'], gpu
selected_names = gpu['names'][:CONFIG['core_gpu_count']]
assert all('AMD Radeon PRO W7900' in name for name in selected_names), selected_names

dataset_entries = [
    line for line in required['training dataset list'].read_text().splitlines()
    if line.strip()
]
assert len(dataset_entries) == 50, len(dataset_entries)

manifest_rows = [entry.split(maxsplit=1) for entry in dataset_entries]
assert all(len(row) == 2 for row in manifest_rows), manifest_rows[:3]
dataset_paths = []
for _, raw_path in manifest_rows:
    dataset_path = Path(raw_path).expanduser()
    if not dataset_path.exists():
        try:
            relative = dataset_path.relative_to('/workspace/RoboTwin')
        except ValueError:
            pass
        else:
            relocated = ROBOTWIN / relative
            if relocated.exists():
                dataset_path = relocated
    assert dataset_path.is_dir(), f'Missing manifest dataset: {dataset_path}'
    dataset_paths.append(dataset_path.resolve())
assert len(set(dataset_paths)) == 50
sample_dataset = next(
    path for path in dataset_paths if path.name == 'adjust_bottle_joint_v30'
)
sample_info_path = sample_dataset / 'meta/info.json'
sample_info = json.loads(sample_info_path.read_text())
sample_features = sample_info['features']

import yaml
training_config = yaml.safe_load(required['LoRA training YAML'].read_text())
assert training_config['data']['cameras'] == ['camera_top', 'camera_wrist_left', 'camera_wrist_right']
assert all(
    any(camera in feature_name for feature_name in sample_features)
    for camera in training_config['data']['cameras']
)
assert sample_features['observation.state']['shape'][-1] == 30
assert sample_features['action']['shape'][-1] == 30
semantic_groups = {}
for item in training_config['data']['joints']:
    semantic_groups.update(ast.literal_eval(item))
assert semantic_groups == {
    'arm.position': 14, 'end.position': 14, 'effector.position': 2,
}
assert training_config['train']['max_state_dim'] == 55
assert training_config['train']['max_action_dim'] == 55
norm_stats = json.loads(required['normalization stats'].read_text())
def contains_key(value, target):
    if isinstance(value, dict):
        return target in value or any(contains_key(child, target) for child in value.values())
    if isinstance(value, list):
        return any(contains_key(child, target) for child in value)
    return False
assert all(contains_key(norm_stats, group) for group in semantic_groups)
assert training_config['train']['use_lora'] is True
assert training_config['train']['lora_scope'] == 'action_expert'
assert training_config['train']['lora_rank'] == 8
assert training_config['train']['lora_alpha'] == 16
assert training_config['train']['seed'] == CONFIG['training_seed']

PATHS['logs'].mkdir(parents=True, exist_ok=False)
(RUNTIME / 'tmp').mkdir(parents=True, exist_ok=True)
run_manifest = {
    'schema_version': '1.0.0',
    'config': CONFIG,
    'companion_commit': companion_commit,
    'companion_checkout_verified': companion_checkout_verified,
    'robotwin_commit': robotwin_commit,
    'lingbot_vla_commit': lingbot_commit,
    'gpu': gpu,
    'dataset_count': len(dataset_entries),
    'sample_dataset': str(sample_dataset),
    'sample_features': sample_features,
    'semantic_groups': semantic_groups,
    'model_capacity': {
        'max_state_dim': training_config['train']['max_state_dim'],
        'max_action_dim': training_config['train']['max_action_dim'],
    },
    'input_audit': input_audit,
    'resolved_seed_matching_available': False,
}
write_json_artifact(PATHS['manifest'], run_manifest)
EVAL_RESULT_ROOT.mkdir(parents=True, exist_ok=True)
print('Preflight passed:', PATHS['manifest'])


## 2. Start the official base-model server

The base checkpoint is the initialization point for training and the deployment reference. The server owns GPU 0 and port 13400; the following cell writes a fresh log and waits for `/healthz` before evaluation.


In [ ]:
server_log = PATHS['logs'] / 'base_server.log'
require_fresh_paths([server_log])
server_command = [
    'bash', str(TRAINING_ROOT / 'scripts/launch_official_server.sh'),
    '0', str(PORT), str(server_log), 'False', str(MODEL),
]
server_log_handle = server_log.open('xb')
SERVER_PROCESS = subprocess.Popen(
    server_command, cwd=ROBOTWIN, stdout=server_log_handle,
    stderr=subprocess.STDOUT, start_new_session=True,
)
print('Base server command:', command_text(server_command))
for _ in range(600):
    if SERVER_PROCESS.poll() is not None:
        raise RuntimeError(f'Base server exited; inspect {server_log}')
    try:
        urllib.request.urlopen(f'http://127.0.0.1:{PORT}/healthz', timeout=2).read()
        print('Base server ready on port', PORT)
        break
    except Exception:
        time.sleep(2)
else:
    raise TimeoutError(f'Base server did not become ready; inspect {server_log}')


## 3. Closed-loop evaluation protocol

Use `adjust_bottle` / `demo_clean`, `expert_check=true`, `accept_expert_info_on_failure=true`, MPLib, EGL, ten episodes, and video enabled for the required base and merged runs. The pinned evaluator may advance its internal seed after an unstable setup, but its current log exposes only aggregate `k/n`. Keep the raw log and generated videos; do not infer matched episode outcomes from episode indices.


### 3.1 Base checkpoint as a pipeline reference

The official base model validates serving, observations, action transport, simulation, and scoring. A modest success count does not by itself mean deployment is broken, and ten episodes are not a stable policy benchmark.


In [ ]:
EVAL_VIDEO_LOG = True
BENCHMARK_VIDEO = False
eval_command = build_single_task_eval_command(
    python=PYTHON, robotwin_root=ROBOTWIN,
    task_name=CONFIG['task_name'], task_config=CONFIG['task_config'],
    episodes=CONFIG['eval_episodes'], port=PORT, video=EVAL_VIDEO_LOG,
)
eval_env = os.environ.copy()
eval_env.update(
    ROBOTWIN_DISABLE_CUROBO='1', ROBOTWIN_EE_PLANNER='mplib',
    PYOPENGL_PLATFORM='egl', PYTHONUNBUFFERED='1',
)
benchmark_env = eval_env.copy()
print('Base evaluation command:', command_text(eval_command))
base_video_before = snapshot_files(EVAL_RESULT_ROOT)
base_eval_started = time.perf_counter()
base_eval_text = run_command_logged(
    eval_command, cwd=ROBOTWIN, env=eval_env,
    log_path=PATHS['base_eval_log'],
)
base_eval_elapsed = time.perf_counter() - base_eval_started
BASE_RESULT = parse_final_success_rate(
    base_eval_text, condition='base', log_path=PATHS['base_eval_log']
)
BASE_VIDEO_ARTIFACTS = archive_changed_files(
    EVAL_RESULT_ROOT, base_video_before, PATHS['base_videos']
)
print('Base aggregate result:', BASE_RESULT)
print('Base video artifacts:', BASE_VIDEO_ARTIFACTS)
print(f'Base evaluation elapsed: {base_eval_elapsed / 60:.2f} min')


### 3.2 Optional official RoboTwin checkpoint

This checkpoint has already been post-trained for RoboTwin and is useful as a competition reference. It is not the starting point for the base-to-LoRA exercise. The code path is retained but disabled by default and uses an independent log/result identity.


In [ ]:
RUN_OFFICIAL_ROBOTWIN_CHECKPOINT = False
ROBOTWIN_CHECKPOINT_ROOT = Path('/models/robotwin-persistent/models/robbyant_lingbot-vla-v2-6b-robotwin')
ROBOTWIN_CHECKPOINT = ROBOTWIN_CHECKPOINT_ROOT
if not (ROBOTWIN_CHECKPOINT / 'model.safetensors.index.json').exists():
    ROBOTWIN_CHECKPOINT = ROBOTWIN_CHECKPOINT_ROOT / 'checkpoints/global_step_50000/hf_ckpt'
ROBOTWIN_CHECKPOINT_CONFIG = ROBOTWIN_CHECKPOINT_ROOT / 'lingbotvla_cli.yaml'

if RUN_OFFICIAL_ROBOTWIN_CHECKPOINT:
    audit_hf_checkpoint(ROBOTWIN_CHECKPOINT)
    for process_name, handle_name in (
        ('SERVER_PROCESS', 'server_log_handle'),
        ('ROBOTWIN_CHECKPOINT_SERVER_PROCESS', 'checkpoint_server_handle'),
    ):
        process = globals().get(process_name)
        if process is not None and process.poll() is None:
            os.killpg(os.getpgid(process.pid), signal.SIGTERM)
            process.wait(timeout=30)
            handle = globals().get(handle_name)
            if handle is not None and not handle.closed:
                handle.close()
    checkpoint_server_log = PATHS['logs'] / 'official_robotwin_server.log'
    require_fresh_paths([checkpoint_server_log])
    checkpoint_server_command = [
        'bash', str(TRAINING_ROOT / 'scripts/launch_official_server.sh'),
        '0', str(PORT), str(checkpoint_server_log), 'False', str(ROBOTWIN_CHECKPOINT),
    ]
    checkpoint_env = eval_env.copy()
    checkpoint_env['LINGBOTVLA_TRAINING_CONFIG'] = str(ROBOTWIN_CHECKPOINT_CONFIG)
    checkpoint_server_handle = checkpoint_server_log.open('xb')
    ROBOTWIN_CHECKPOINT_SERVER_PROCESS = subprocess.Popen(
        checkpoint_server_command, cwd=ROBOTWIN, env=checkpoint_env,
        stdout=checkpoint_server_handle, stderr=subprocess.STDOUT,
        start_new_session=True,
    )
    for _ in range(600):
        if ROBOTWIN_CHECKPOINT_SERVER_PROCESS.poll() is not None:
            raise RuntimeError(f'Checkpoint server exited; inspect {checkpoint_server_log}')
        try:
            urllib.request.urlopen(f'http://127.0.0.1:{PORT}/healthz', timeout=2).read()
            break
        except Exception:
            time.sleep(2)
    official_eval_log = PATHS['logs'] / 'official_robotwin_evaluation.log'
    official_video_before = snapshot_files(EVAL_RESULT_ROOT)
    official_text = run_command_logged(
        eval_command, cwd=ROBOTWIN, env=checkpoint_env,
        log_path=official_eval_log,
    )
    OFFICIAL_ROBOTWIN_RESULT = parse_final_success_rate(
        official_text, condition='official_robotwin', log_path=official_eval_log
    )
    OFFICIAL_VIDEO_ARTIFACTS = archive_changed_files(
        EVAL_RESULT_ROOT, official_video_before, PATHS['official_videos']
    )
    print(OFFICIAL_ROBOTWIN_RESULT)
else:
    print('SKIPPED: optional official RoboTwin checkpoint evaluation')


### 3.3 Optional official-base 100 tasks × 10 episodes

This is a long 1,000-episode benchmark over clean and randomized tasks. It launches one server per selected GPU, defaults to the 8-GPU competition profile, and is disabled unless you explicitly opt in.


In [ ]:
RUN_OFFICIAL_FULL_BENCHMARK = False
BENCHMARK_GPU_COUNT = CONFIG['competition_gpu_count']
BENCHMARK_EPISODES = 10
BENCHMARK_RUN_NAME = f"{CONFIG['run_name']}-official-both100x10-{BENCHMARK_GPU_COUNT}gpu"
BENCHMARK_RESUME = False
if RUN_OFFICIAL_FULL_BENCHMARK:
    if BENCHMARK_GPU_COUNT not in (4, 8):
        raise ValueError('BENCHMARK_GPU_COUNT must be 4 or 8')
    for process_name, handle_name in (
        ('SERVER_PROCESS', 'server_log_handle'),
        ('ROBOTWIN_CHECKPOINT_SERVER_PROCESS', 'checkpoint_server_handle'),
    ):
        process = globals().get(process_name)
        if process is not None and process.poll() is None:
            os.killpg(os.getpgid(process.pid), signal.SIGTERM)
            process.wait(timeout=30)
            handle = globals().get(handle_name)
            if handle is not None and not handle.closed:
                handle.close()
    benchmark_command = [
        PYTHON, str(TRAINING_ROOT / 'scripts/run_clean_benchmark.py'),
        '--gpu-count', str(BENCHMARK_GPU_COUNT), '--episodes', str(BENCHMARK_EPISODES),
        '--expert-check', '--accept-expert-info-on-failure',
        ('--video' if BENCHMARK_VIDEO else '--no-video'),
        '--run-name', BENCHMARK_RUN_NAME,
        '--runtime-dir', str(RUNTIME), '--robotwin-root', str(ROBOTWIN),
    ]
    benchmark_run_root = RUNTIME / 'outputs' / BENCHMARK_RUN_NAME
    if BENCHMARK_RESUME:
        benchmark_command.append('--resume')
    else:
        require_fresh_paths([benchmark_run_root])
    subprocess.run(benchmark_command, cwd=ROBOTWIN, env=benchmark_env, check=True)
else:
    print('SKIPPED: optional official-base 100-task benchmark')


### 3.4 Optional official RoboTwin-checkpoint benchmark

This aligned long benchmark changes the checkpoint identity while preserving the scheduler and task set. Keep its run name and outputs separate from the official-base benchmark.


In [ ]:
RUN_ROBOTWIN_FULL_BENCHMARK = False
ROBOTWIN_BENCHMARK_RUN_NAME = f"{CONFIG['run_name']}-robotwin-both100x10-{BENCHMARK_GPU_COUNT}gpu"
ROBOTWIN_BENCHMARK_RESUME = False
if RUN_ROBOTWIN_FULL_BENCHMARK:
    audit_hf_checkpoint(ROBOTWIN_CHECKPOINT)
    robotwin_benchmark_env = benchmark_env.copy()
    robotwin_benchmark_env['LINGBOTVLA_TRAINING_CONFIG'] = str(ROBOTWIN_CHECKPOINT_CONFIG)
    robotwin_benchmark_command = [
        PYTHON, str(TRAINING_ROOT / 'scripts/run_clean_benchmark.py'),
        '--gpu-count', str(BENCHMARK_GPU_COUNT), '--episodes', str(BENCHMARK_EPISODES),
        '--model-path', str(ROBOTWIN_CHECKPOINT), '--expert-check',
        '--accept-expert-info-on-failure',
        ('--video' if BENCHMARK_VIDEO else '--no-video'),
        '--run-name', ROBOTWIN_BENCHMARK_RUN_NAME,
        '--runtime-dir', str(RUNTIME), '--robotwin-root', str(ROBOTWIN),
    ]
    robotwin_benchmark_root = RUNTIME / 'outputs' / ROBOTWIN_BENCHMARK_RUN_NAME
    if ROBOTWIN_BENCHMARK_RESUME:
        robotwin_benchmark_command.append('--resume')
    else:
        require_fresh_paths([robotwin_benchmark_root])
    subprocess.run(robotwin_benchmark_command, cwd=ROBOTWIN, env=robotwin_benchmark_env, check=True)
else:
    print('SKIPPED: optional official-RoboTwin 100-task benchmark')


## 4. Action-expert LoRA, DCP merge, and inference

The required training profile uses four W7900 GPUs and 100 optimizer steps. It is a pipeline smoke test, not evidence that policy quality generally improves. The 8-GPU profile is retained for competition-scale extension work.


### 4.1 Train the action-expert LoRA adapter

The pinned YAML applies rank-8, alpha-16 LoRA to the action expert's `q_proj`, `k_proj`, `v_proj`, and `o_proj` modules. With micro batch 1, four data-parallel shards, and global batch 4, gradient accumulation is 1. FSDP2 shards the trainable state, and seed 42 fixes the training-side random schedule.


In [ ]:
for process_name, handle_name in (
    ('SERVER_PROCESS', 'server_log_handle'),
    ('ROBOTWIN_CHECKPOINT_SERVER_PROCESS', 'checkpoint_server_handle'),
):
    process = globals().get(process_name)
    if process is not None and process.poll() is None:
        os.killpg(os.getpgid(process.pid), signal.SIGTERM)
        process.wait(timeout=30)
        handle = globals().get(handle_name)
        if handle is not None and not handle.closed:
            handle.close()

GPU_COUNT = CONFIG['core_gpu_count']  # Change explicitly to competition_gpu_count for 8 GPUs.
TRAIN_STEPS = CONFIG['lora_steps']
LORA_GLOBAL_BATCH_SIZE = GPU_COUNT
LORA_MICRO_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = LORA_GLOBAL_BATCH_SIZE // (GPU_COUNT * LORA_MICRO_BATCH_SIZE)
training_yaml = TRAINING_ROOT / 'training/lingbotvla_cli.yaml'
training_output = PATHS['lora_output']
require_fresh_paths([training_output])
train_command = [
    PYTHON, '-m', 'torch.distributed.run', '--standalone',
    f'--nproc-per-node={GPU_COUNT}', '-m', 'tasks.vla.train_lingbotvla',
    str(training_yaml),
    '--train.data_parallel_shard_size', str(GPU_COUNT),
    '--train.micro_batch_size', str(LORA_MICRO_BATCH_SIZE),
    '--train.gradient_accumulation_steps', str(GRADIENT_ACCUMULATION_STEPS),
    '--train.global_batch_size', str(LORA_GLOBAL_BATCH_SIZE),
    '--train.max_steps', str(TRAIN_STEPS),
    '--train.save_steps', str(TRAIN_STEPS),
    '--train.seed', str(CONFIG['training_seed']),
    '--train.output_dir', str(training_output),
]
train_env = os.environ.copy()
train_env['HIP_VISIBLE_DEVICES'] = ','.join(str(i) for i in range(GPU_COUNT))
train_env.pop('ROCR_VISIBLE_DEVICES', None)
train_env.pop('CUDA_VISIBLE_DEVICES', None)
print('LoRA command:', command_text(train_command))
training_started = time.perf_counter()
run_command_logged(
    train_command, cwd=SOURCE_ROOT, env=train_env,
    log_path=PATHS['logs'] / 'lora_training.log',
)
training_elapsed = time.perf_counter() - training_started
print(f'LoRA training elapsed: {training_elapsed / 60:.2f} min')


### 4.2 Merge the LoRA DCP checkpoint

The training checkpoint is distributed DCP state, not a directly serveable model. The merge script reconstructs the adapter, verifies rank/alpha, applies it to the exact base checkpoint, and writes a Hugging Face checkpoint. The next cell audits config, processor/tokenizer files, weight index, and every referenced shard.


In [ ]:
checkpoint = training_output / f'checkpoints/global_step_{TRAIN_STEPS}'
MERGED = training_output / f'merged_checkpoint/global_step_{TRAIN_STEPS}/hf_ckpt'
assert checkpoint.is_dir(), f'Missing DCP checkpoint: {checkpoint}'
require_fresh_paths([MERGED])
merge_command = [
    PYTHON, str(TRAINING_ROOT / 'scripts/merge_lora_dcp.py'),
    '--checkpoint', str(checkpoint), '--training-output', str(training_output),
    '--base-model', str(MODEL), '--output', str(MERGED),
    '--rank', '8', '--alpha', '16',
]
print('LoRA merge command:', command_text(merge_command))
run_command_logged(
    merge_command, cwd=ROBOTWIN, env=os.environ.copy(),
    log_path=PATHS['logs'] / 'lora_merge.log',
)
MERGED_AUDIT = audit_hf_checkpoint(MERGED)
print(json.dumps(MERGED_AUDIT, indent=2))


### 4.3 Start the merged LoRA server

Reuse port 13400 and the same runtime flags. Checkpoint identity should be the intended change between the base and merged evaluations.


In [ ]:
merged_log = PATHS['logs'] / 'merged_server.log'
require_fresh_paths([merged_log])
merged_server_command = [
    'bash', str(TRAINING_ROOT / 'scripts/launch_official_server.sh'),
    '0', str(PORT), str(merged_log), 'False', str(MERGED),
]
merged_log_handle = merged_log.open('xb')
MERGED_SERVER_PROCESS = subprocess.Popen(
    merged_server_command, cwd=ROBOTWIN, stdout=merged_log_handle,
    stderr=subprocess.STDOUT, start_new_session=True,
)
print('Merged server command:', command_text(merged_server_command))
for _ in range(600):
    if MERGED_SERVER_PROCESS.poll() is not None:
        raise RuntimeError(f'Merged server exited; inspect {merged_log}')
    try:
        urllib.request.urlopen(f'http://127.0.0.1:{PORT}/healthz', timeout=2).read()
        print('Merged server ready on port', PORT)
        break
    except Exception:
        time.sleep(2)
else:
    raise TimeoutError(f'Merged server did not become ready; inspect {merged_log}')


### 4.4 Evaluate the merged LoRA checkpoint

Repeat the same ten-episode command and preserve a separate log. Because the current companion does not expose final per-resolved-seed outcomes, compare the two `k/n` values only as independent small-run observations. A larger merged count is not a matched improvement claim.


In [ ]:
print('Merged evaluation command:', command_text(eval_command))
merged_video_before = snapshot_files(EVAL_RESULT_ROOT)
merged_eval_started = time.perf_counter()
merged_eval_text = run_command_logged(
    eval_command, cwd=ROBOTWIN, env=eval_env,
    log_path=PATHS['merged_eval_log'],
)
merged_eval_elapsed = time.perf_counter() - merged_eval_started
MERGED_RESULT = parse_final_success_rate(
    merged_eval_text, condition='merged_lora',
    log_path=PATHS['merged_eval_log'],
)
MERGED_VIDEO_ARTIFACTS = archive_changed_files(
    EVAL_RESULT_ROOT, merged_video_before, PATHS['merged_videos']
)
print('Merged aggregate result:', MERGED_RESULT)
print('Merged video artifacts:', MERGED_VIDEO_ARTIFACTS)
print(f'Merged evaluation elapsed: {merged_eval_elapsed / 60:.2f} min')


### 4.5 Optional merged-LoRA 100 tasks × 10 episodes

The long benchmark is retained with its own opt-in and run identity. It is independent from the required 10+10 smoke path.


In [ ]:
RUN_LORA_FULL_BENCHMARK = False
LORA_BENCHMARK_GPU_COUNT = CONFIG['competition_gpu_count']
LORA_BENCHMARK_RUN_NAME = f"{CONFIG['run_name']}-lora-both100x10-{LORA_BENCHMARK_GPU_COUNT}gpu"
LORA_BENCHMARK_RESUME = False
if RUN_LORA_FULL_BENCHMARK:
    if 'MERGED_SERVER_PROCESS' in globals() and MERGED_SERVER_PROCESS.poll() is None:
        os.killpg(os.getpgid(MERGED_SERVER_PROCESS.pid), signal.SIGTERM)
        MERGED_SERVER_PROCESS.wait(timeout=30)
        merged_log_handle.close()
    lora_benchmark_command = [
        PYTHON, str(TRAINING_ROOT / 'scripts/run_clean_benchmark.py'),
        '--gpu-count', str(LORA_BENCHMARK_GPU_COUNT), '--episodes', '10',
        '--model-path', str(MERGED), '--expert-check',
        '--accept-expert-info-on-failure',
        ('--video' if BENCHMARK_VIDEO else '--no-video'),
        '--run-name', LORA_BENCHMARK_RUN_NAME,
        '--runtime-dir', str(RUNTIME), '--robotwin-root', str(ROBOTWIN),
    ]
    lora_benchmark_root = RUNTIME / 'outputs' / LORA_BENCHMARK_RUN_NAME
    if LORA_BENCHMARK_RESUME:
        lora_benchmark_command.append('--resume')
    else:
        require_fresh_paths([lora_benchmark_root])
    subprocess.run(lora_benchmark_command, cwd=ROBOTWIN, env=benchmark_env, check=True)
else:
    print('SKIPPED: optional merged-LoRA 100-task benchmark')


### 4.6 Stop the merged server and write the scoped evidence card

The evidence card records each aggregate count and Wilson interval, the observed rate difference, pinned identities, and the explicit fact that matched outcomes are unavailable. Keep that limitation even when the merged count is higher.


In [ ]:
if 'MERGED_SERVER_PROCESS' in globals() and MERGED_SERVER_PROCESS.poll() is None:
    os.killpg(os.getpgid(MERGED_SERVER_PROCESS.pid), signal.SIGTERM)
    MERGED_SERVER_PROCESS.wait(timeout=30)
    merged_log_handle.close()
    print('Merged model server stopped')

AGGREGATE_EVIDENCE = build_aggregate_evidence_card(
    BASE_RESULT,
    MERGED_RESULT,
    identities={
        'run_name': CONFIG['run_name'],
        'task_name': CONFIG['task_name'],
        'task_config': CONFIG['task_config'],
        'protocol': 'lingbot_vla_v2',
        'cli_seed': 0,
        'eval_episodes': CONFIG['eval_episodes'],
        'base_checkpoint': str(MODEL),
        'merged_lora_checkpoint': str(MERGED),
        'training_seed': CONFIG['training_seed'],
        'lora_steps': TRAIN_STEPS,
        'lora_rank': 8,
        'lora_alpha': 16,
    },
    video_artifacts={
        'base': BASE_VIDEO_ARTIFACTS,
        'merged_lora': MERGED_VIDEO_ARTIFACTS,
    },
)
write_json_artifact(PATHS['aggregate_evidence'], AGGREGATE_EVIDENCE)
print(json.dumps(AGGREGATE_EVIDENCE, indent=2))
print('IMPORTANT: matched outcomes are unavailable with the pinned companion.')
print('Do not claim that the observed aggregate difference is a resolved-seed-matched improvement.')


## 5. Optional full-parameter SFT, conversion, and inference

Full SFT is a separate training path: `use_lora=false` updates all trainable parameters and the retained default uses full depth/video teacher supervision. It is substantially more expensive in time, memory, and storage, so the section is visible but disabled by default. Enabling it runs training, DCP conversion, serving, and evaluation rather than stopping at loss.


In [ ]:
subprocess.run(['df', '-h', '/', '/workspace'], check=True)
outputs_root = RUNTIME / 'outputs'
subprocess.run(['du', '-d1', '-h', str(outputs_root)], check=True)
cleanup_candidates = sorted(
    set(outputs_root.glob('lora_*'))
    | set(outputs_root.glob('reproduction_*'))
    | set(outputs_root.glob('full_sft_*'))
)
print('Dry-run cleanup candidates; this notebook does not delete them:')
for candidate in cleanup_candidates:
    print(' -', candidate)
print('Archive required checkpoints, logs, videos, and evidence before manual deletion.')


### 5.1 Run full-parameter SFT

The launcher derives accumulation from global batch 256, GPU count, and micro batch. The four-W7900 reference uses micro batch 16 with accumulation 4 and full reshard after forward. Review the dry-run storage list before enabling this cell.


In [ ]:
RUN_FULL_SFT = False
FULL_SFT_GPU_COUNT = CONFIG['core_gpu_count']
FULL_SFT_STEPS = 10
FULL_SFT_SAVE_STEPS = FULL_SFT_STEPS
FULL_SFT_TEACHER_MODE = 'full'
FULL_SFT_TIMEOUT_SECONDS = 0
FULL_SFT_GLOBAL_BATCH_SIZE = 256
FULL_SFT_MICRO_BATCH_SIZE = 16
FULL_SFT_ENABLE_FULL_SHARD = True
FULL_SFT_OPTIMIZER = 'adamw'
FULL_SFT_OUTPUT = PATHS['full_sft_output']

if RUN_FULL_SFT:
    assert FULL_SFT_GPU_COUNT in (4, 8)
    assert FULL_SFT_TEACHER_MODE in ('full', 'current_depth', 'none')
    require_fresh_paths([FULL_SFT_OUTPUT])
    gpu_count = int(subprocess.check_output(
        [PYTHON, '-c', 'import torch; print(torch.cuda.device_count())'], text=True
    ).strip())
    assert gpu_count >= FULL_SFT_GPU_COUNT
    full_sft_env = os.environ.copy()
    full_sft_env.update({
        'GPU_COUNT': str(FULL_SFT_GPU_COUNT), 'MAX_STEPS': str(FULL_SFT_STEPS),
        'SAVE_STEPS': str(FULL_SFT_SAVE_STEPS), 'TEACHER_MODE': FULL_SFT_TEACHER_MODE,
        'TIMEOUT_SECONDS': str(FULL_SFT_TIMEOUT_SECONDS),
        'GLOBAL_BATCH_SIZE': str(FULL_SFT_GLOBAL_BATCH_SIZE),
        'MICRO_BATCH_SIZE': str(FULL_SFT_MICRO_BATCH_SIZE),
        'ENABLE_FULL_SHARD': str(FULL_SFT_ENABLE_FULL_SHARD).lower(),
        'OPTIMIZER': FULL_SFT_OPTIMIZER, 'OUTPUT_DIR': str(FULL_SFT_OUTPUT),
    })
    full_sft_command = [
        'bash', str(TRAINING_ROOT / 'training/train_full_sft.sh')
    ]
    print('Full-SFT command:', command_text(full_sft_command))
    run_command_logged(
        full_sft_command, cwd=ROBOTWIN, env=full_sft_env,
        log_path=PATHS['logs'] / 'full_sft_training.log',
    )
else:
    print('SKIPPED: full-parameter SFT is an opt-in high-cost extension')


### 5.2 Pinned companion full-teacher capacity reference

These are competition-reference measurements reported by the pinned companion, not RoboGenesis verification. Steady time is the mean of optimizer steps 2 and 3; VRAM is the maximum sampled across loading, forward, backward, and optimizer update.

| Platform | Micro batch/GPU | Accumulation | Steady StepTime | Peak VRAM/GPU | Result |
|---|---:|---:|---:|---:|---|
| 4 × W7900 48 GiB | 1 | 64 | 245.580 s | 40.329–40.375 GiB | passed |
| 4 × W7900 48 GiB | 2 | 32 | 136.748 s | 41.337–41.349 GiB | passed |
| 4 × W7900 48 GiB | 4 | 16 | 76.010 s | 43.509–43.521 GiB | passed |
| 4 × W7900 48 GiB | 8 | 8 | 54.031 s | 47.729–47.761 GiB | passed; limited headroom |
| 4 × W7900 48 GiB | 16 | 4 | 41.392 s | 47.890–47.975 GiB | passed; nearly full |
| 4 × W7900 48 GiB | 32 | 2 | — | 47.648–47.956 GiB | HIP OOM |
| 8 × W7900 48 GiB | 1 | 32 | 143.665 s | 27.520–27.558 GiB | computed |
| 8 × W7900 48 GiB | 2 | 16 | 80.319 s | 27.818–27.832 GiB | passed |
| 8 × W7900 48 GiB | 4 | 8 | 47.055 s | 29.498–29.523 GiB | passed |
| 8 × W7900 48 GiB | 8 | 4 | 27.108 s | 33.004–33.386 GiB | passed |
| 8 × W7900 48 GiB | 16 | 2 | 24.682 s | 41.344–41.356 GiB | passed |
| 8 × W7900 48 GiB | 32 | 1 | — | 47.878–47.947 GiB | HIP OOM |


### 5.3 Convert full-SFT DCP to a serving checkpoint

Full SFT has no adapter to merge. Conversion gathers the distributed model state, filters training-only teacher heads according to the pinned implementation, and writes a directly loadable Hugging Face checkpoint.


In [ ]:
FULL_SFT_CHECKPOINT = FULL_SFT_OUTPUT / f'checkpoints/global_step_{FULL_SFT_STEPS}'
FULL_SFT_MODEL = FULL_SFT_OUTPUT / f'merged_checkpoint/global_step_{FULL_SFT_STEPS}/hf_ckpt'
if RUN_FULL_SFT:
    assert FULL_SFT_CHECKPOINT.is_dir(), f'Missing checkpoint: {FULL_SFT_CHECKPOINT}'
    require_fresh_paths([FULL_SFT_MODEL])
    full_sft_convert_command = [
        PYTHON, str(TRAINING_ROOT / 'scripts/convert_full_sft_dcp.py'),
        '--checkpoint', str(FULL_SFT_CHECKPOINT),
        '--training-output', str(FULL_SFT_OUTPUT),
        '--output', str(FULL_SFT_MODEL),
    ]
    run_command_logged(
        full_sft_convert_command, cwd=ROBOTWIN, env=os.environ.copy(),
        log_path=PATHS['logs'] / 'full_sft_conversion.log',
    )
    FULL_SFT_MODEL_AUDIT = audit_hf_checkpoint(FULL_SFT_MODEL)
    print(json.dumps(FULL_SFT_MODEL_AUDIT, indent=2))
else:
    print('SKIPPED: no full-SFT DCP checkpoint was produced')


### 5.4 Start the converted full-SFT server

This uses the same serving entry point as base and merged LoRA, but an independent checkpoint and log identity.


In [ ]:
if RUN_FULL_SFT:
    full_sft_server_log = PATHS['logs'] / 'full_sft_server.log'
    require_fresh_paths([full_sft_server_log])
    full_sft_server_command = [
        'bash', str(TRAINING_ROOT / 'scripts/launch_official_server.sh'),
        '0', str(PORT), str(full_sft_server_log), 'False', str(FULL_SFT_MODEL),
    ]
    full_sft_server_handle = full_sft_server_log.open('xb')
    FULL_SFT_SERVER_PROCESS = subprocess.Popen(
        full_sft_server_command, cwd=ROBOTWIN, stdout=full_sft_server_handle,
        stderr=subprocess.STDOUT, start_new_session=True,
    )
    for _ in range(600):
        if FULL_SFT_SERVER_PROCESS.poll() is not None:
            raise RuntimeError(f'Full-SFT server exited; inspect {full_sft_server_log}')
        try:
            urllib.request.urlopen(f'http://127.0.0.1:{PORT}/healthz', timeout=2).read()
            print('Full-SFT server ready on port', PORT)
            break
        except Exception:
            time.sleep(2)
    else:
        raise TimeoutError(f'Full-SFT server did not become ready; inspect {full_sft_server_log}')
else:
    print('SKIPPED: full-SFT server was not started')


### 5.5 Evaluate the converted full-SFT checkpoint

Run the same aggregate closed-loop protocol only when full SFT and conversion completed. Do not pool this result into the base/LoRA denominator.


In [ ]:
if RUN_FULL_SFT:
    full_sft_eval_log = PATHS['logs'] / 'full_sft_evaluation.log'
    full_sft_video_before = snapshot_files(EVAL_RESULT_ROOT)
    full_sft_eval_text = run_command_logged(
        eval_command, cwd=ROBOTWIN, env=eval_env,
        log_path=full_sft_eval_log,
    )
    FULL_SFT_RESULT = parse_final_success_rate(
        full_sft_eval_text, condition='full_sft', log_path=full_sft_eval_log
    )
    FULL_SFT_VIDEO_ARTIFACTS = archive_changed_files(
        EVAL_RESULT_ROOT, full_sft_video_before, PATHS['full_sft_videos']
    )
    print('Full-SFT aggregate result:', FULL_SFT_RESULT)
    print('Full-SFT video artifacts:', FULL_SFT_VIDEO_ARTIFACTS)
else:
    print('SKIPPED: no converted full-SFT model is being served')


### 5.6 Optional full-SFT 100 tasks × 10 episodes

This retained competition extension is disabled by default and uses a separate benchmark run name.


In [ ]:
RUN_FULL_SFT_FULL_BENCHMARK = False
FULL_SFT_BENCHMARK_GPU_COUNT = CONFIG['competition_gpu_count']
FULL_SFT_BENCHMARK_RUN_NAME = f"{CONFIG['run_name']}-full-sft-both100x10-{FULL_SFT_BENCHMARK_GPU_COUNT}gpu"
FULL_SFT_BENCHMARK_RESUME = False
if RUN_FULL_SFT and RUN_FULL_SFT_FULL_BENCHMARK:
    if 'FULL_SFT_SERVER_PROCESS' in globals() and FULL_SFT_SERVER_PROCESS.poll() is None:
        os.killpg(os.getpgid(FULL_SFT_SERVER_PROCESS.pid), signal.SIGTERM)
        FULL_SFT_SERVER_PROCESS.wait(timeout=30)
        full_sft_server_handle.close()
    full_sft_benchmark_command = [
        PYTHON, str(TRAINING_ROOT / 'scripts/run_clean_benchmark.py'),
        '--gpu-count', str(FULL_SFT_BENCHMARK_GPU_COUNT), '--episodes', '10',
        '--model-path', str(FULL_SFT_MODEL), '--expert-check',
        '--accept-expert-info-on-failure',
        ('--video' if BENCHMARK_VIDEO else '--no-video'),
        '--run-name', FULL_SFT_BENCHMARK_RUN_NAME,
        '--runtime-dir', str(RUNTIME), '--robotwin-root', str(ROBOTWIN),
    ]
    full_sft_benchmark_root = RUNTIME / 'outputs' / FULL_SFT_BENCHMARK_RUN_NAME
    if FULL_SFT_BENCHMARK_RESUME:
        full_sft_benchmark_command.append('--resume')
    else:
        require_fresh_paths([full_sft_benchmark_root])
    subprocess.run(full_sft_benchmark_command, cwd=ROBOTWIN, env=benchmark_env, check=True)
else:
    print('SKIPPED: optional full-SFT 100-task benchmark')


### 5.7 Cleanup and artifact inventory

Stop only the server process created by this kernel. The final inventory lists DCP, serving checkpoints, logs, and evidence without deleting them. The completed learning outcome is the ability to run and distinguish LoRA and full SFT; policy-quality claims require stronger evaluation evidence.


In [ ]:
if 'FULL_SFT_SERVER_PROCESS' in globals() and FULL_SFT_SERVER_PROCESS.poll() is None:
    os.killpg(os.getpgid(FULL_SFT_SERVER_PROCESS.pid), signal.SIGTERM)
    FULL_SFT_SERVER_PROCESS.wait(timeout=30)
    full_sft_server_handle.close()
    print('Full-SFT model server stopped')
else:
    print('No full-SFT server started by this notebook kernel')

final_inventory = {
    'run_root': str(PATHS['run_root']),
    'manifest': str(PATHS['manifest']),
    'lora_dcp': str(checkpoint) if 'checkpoint' in globals() else None,
    'merged_lora_model': str(MERGED) if 'MERGED' in globals() else None,
    'aggregate_evidence': str(PATHS['aggregate_evidence']) if PATHS['aggregate_evidence'].exists() else None,
    'base_videos': BASE_VIDEO_ARTIFACTS,
    'merged_lora_videos': MERGED_VIDEO_ARTIFACTS,
    'full_sft_dcp': str(FULL_SFT_CHECKPOINT) if RUN_FULL_SFT else None,
    'full_sft_model': str(FULL_SFT_MODEL) if RUN_FULL_SFT else None,
    'full_sft_videos': FULL_SFT_VIDEO_ARTIFACTS if RUN_FULL_SFT else None,
    'resolved_seed_matching_available': False,
}
print(json.dumps(final_inventory, indent=2))
